In [20]:
import os, shutil, math

In [21]:
parent_dir = '/Users/shubhamjamdade/Desktop/MD Snapshot/Convert CIF to data.adsorbate/Test/'
#path_data_adsorbate_template = '/Users/shubhamjamdade/Desktop/MD Snapshot/Convert CIF to data.adsorbate/General Files/data.adsorbate_template/'
path_general_files = '/Users/shubhamjamdade/Desktop/MD Snapshot/Convert CIF to data.adsorbate/General Files/'


In [22]:
for folder in os.listdir(parent_dir):
    path_MOF = os.path.join(parent_dir, folder)
    path_data_adsorbate = os.path.join(path_MOF,'data.adsorbate')
    print(path_MOF)
    for filename in os.listdir(path_MOF):
        if filename.startswith("restart_"):
            path_restart = os.path.join(path_MOF, filename)
            x_O = []
            y_O = []
            z_O = []

            x_H1 = []
            y_H1 = []
            z_H1 = []

            x_H2 = []
            y_H2 = []
            z_H2 = []
            
            d_H1 = []
            d_H2 = []
            
            D_H1 = None
            D_H2 = None

            fr = open(str(path_restart), "rt")

            lines = fr.readlines()
            for line in lines:
                row=line.split()

                if not line.strip():
                   continue

                if len(row) == 7 and row[0] == 'Component:' and row[2] =='Adsorbate':
                    No_of_adsorbates = int(row[3])
#                     No_of_atoms = 3*No_of_adsorbates
#                     No_of_bonds = 2*No_of_adsorbates
#                     No_of_angles = No_of_adsorbates

                if len(row) == 4 and row[0] == 'cell-vector-a:':
                    a = row[1]
                if len(row) == 4 and row[0] == 'cell-vector-b:':   
                    b = row[2]
                if len(row) == 4 and row[0] == 'cell-vector-c:':   
                    c = row[3]
                if len(row) == 6 and row[0] == 'Adsorbate-atom-position:' and row[2] == str(0):
                    x_O.append(row[3]) 
                    y_O.append(row[4]) 
                    z_O.append(row[5]) 
                if len(row) == 6 and row[0] =='Adsorbate-atom-position:' and row[2] == str(1) :
                    x_H1.append(row[3]) 
                    y_H1.append(row[4]) 
                    z_H1.append(row[5]) 
                if len(row) == 6 and row[0] =='Adsorbate-atom-position:' and row[2] == str(2) :
                    x_H2.append(row[3]) 
                    y_H2.append(row[4]) 
                    z_H2.append(row[5]) 
                if len(row) == 6 and row[0] =='Adsorbate-atom-position:' and row[2] == str(3) : 
                    
                    D_H1 = ((float(x_O[-1])-float(x_H1[-1]))**2 + (float(y_O[-1])-float(y_H1[-1]))**2 + (float(z_O[-1])-float(z_H1[-1]))**2)**0.5
                    D_H2 = ((float(x_O[-1])-float(x_H2[-1]))**2 + (float(y_O[-1])-float(y_H2[-1]))**2 + (float(z_O[-1])-float(z_H2[-1]))**2)**0.5
                    
                    d_H1.append(D_H1)
                    d_H2.append(D_H2)
                    
            with open(str(path_data_adsorbate), 'w') as file:

                shutil.copyfile(str(path_general_files)+'data.adsorbate_template', str(path_data_adsorbate))

            fin = open(str(path_data_adsorbate), 'r')
            data = fin.read()
            data = data.replace('vector_A', str(a))
            data = data.replace('vector_B', str(b))
            data = data.replace('vector_C', str(c))
            data = data.replace('Angle_AB', str(c))
            data = data.replace('Angle_AC', str(c))
            data = data.replace('Angle_BC', str(c))
            
            No_of_atoms = 3*No_of_adsorbates
            No_of_bonds = 2*No_of_adsorbates
            No_of_angles = No_of_adsorbates

            data = data.replace('No_of_atoms', str(No_of_atoms))
            data = data.replace('No_of_bonds', str(No_of_bonds))
            data = data.replace('No_of_angles', str(No_of_angles))

            fin.close()

            fin = open(str(path_data_adsorbate), 'w')

            fin.write(data)

            fin.close()
            
            ##
            
            fd = open(str(path_data_adsorbate), 'r')

            lines_d = fd.readlines()

            with open(str(path_data_adsorbate), 'a') as file:

                for line in lines_d:
                    row=line.split()
                    if not line.strip():
                         continue

                    if len(row) == 8 and row[1] == 'harmonic' and row[7] == 'H_TIP4P':

                        j = 0

                        for i in range(1, int(No_of_adsorbates)+1):
                            if j == 0 :
                                file.write('\n\nAtoms\n')

                            Atoms_line_1 = os.linesep + str(j+1) + '   ' + str(i) + '   ' + str(1) + '   ' + str(-1.040) + '   ' + str(x_O[i-1])  + '   ' + str(y_O[i-1])+ '   ' + str(z_O[i-1]) 
                            file.write(str(Atoms_line_1))
                            Atoms_line_2 = os.linesep + str(j+2) + '   ' + str(i) + '   ' + str(2) + '   ' + str(0.5200) + '   ' + str(x_H1[i-1])  + '   ' + str(y_H1[i-1])+ '   ' + str(z_H1[i-1])
                            file.write(str(Atoms_line_2))
                            Atoms_line_3 = os.linesep + str(j+3) + '   ' + str(i) + '   ' + str(2) + '   ' + str(0.5200) + '   ' + str(x_H2[i-1])  + '   ' + str(y_H2[i-1])+ '   ' + str(z_H2[i-1])
                            file.write(str(Atoms_line_3))

                            j = j + 3

                        l = 0

                        for k in range(1, int(No_of_bonds)+1, 2):
                            if k == int(No_of_bonds):
                                break 
                            if l == 0 :
                                file.write('\n\nBonds\n')

                            Bonds_line_1 = os.linesep  + str(k) + '   ' + str(1) + '   ' + str(l+1) + '   ' + str(l+2)  
                            file.write(str(Bonds_line_1))
                            p =  k + 1
                            Bonds_line_2 = os.linesep  + str(p) + '   ' + str(1) + '   ' + str(l+1) + '   ' + str(l+3) 
                            file.write(str(Bonds_line_2))

                            l = l + 3                

                        file.write('\n\nAngles\n')
                        for m in range(1, int(No_of_angles)+1):

                            Angles_line_1 = os.linesep  + str(m) + '   ' + str(1) + '   ' + str(3*m-1) + '   ' + str(3*m-2) + '   ' + str(3*m)  
                            file.write(str(Angles_line_1))



            

/Users/shubhamjamdade/Desktop/MD Snapshot/Convert CIF to data.adsorbate/Test/ATOXEN_111


In [23]:
for value in d_H1: 
    if value > 1:
        index = d_H1.index(str(value))
        print(index)
        
for value in d_H2: 
    if value > 1:
        index = d_H2.index(str(value))
        print(index)